# ttbar Semileptonic — Output Exploration

This notebook is for interactive exploration of the `.coffea` output files.
It complements `make_plots.py` with pandas-friendly access to cutflows,
histogram integrals, and columnar outputs.

**Typical workflow after copying output to laptop:**
```
scp lpc:/path/to/outputs/test_run.coffea outputs/
jupyter lab notebooks/explore_output.ipynb
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mplhep as hep
import coffea.util
import hist
import yaml

plt.style.use(hep.style.CMS)

OUTPUT_FILE = '../outputs/test_run.coffea'   # ← change as needed
LUMI        = 59722.768   # pb^-1, UL18

out = coffea.util.load(OUTPUT_FILE)
print('Keys in output:', list(out.keys()))

## Cutflow

In [ ]:
# pocket-coffea stores cutflows in out['cutflow']
# Structure: {sample: {cut_name: n_events}}

if 'cutflow' in out:
    cutflow_df = pd.DataFrame(out['cutflow']).T
    cutflow_df.index.name = 'sample'
    display(cutflow_df.style.format('{:,.0f}'))
else:
    print('No cutflow found; check pocket-coffea version.')

## List available histograms

In [ ]:
hists = out.get('histograms', {})
print('Histogram variables:', list(hists.keys()))

## Quick single-sample histogram

In [ ]:
var      = 'muon_pt'
sample   = 'TTToSemiLeptonic'
category = 'SR'

h = hists[var][(sample, category, 'nominal')]

fig, ax = plt.subplots(figsize=(7, 5))
h.plot1d(ax=ax, label=sample)
ax.set_xlabel(r'Muon $p_T$ [GeV]')
ax.set_ylabel('Events / bin')
ax.legend()
hep.cms.label('Simulation Preliminary', ax=ax)
plt.tight_layout()

## Systematic variations

In [ ]:
# List all (sample, category, variation) keys for a given variable
var = 'muon_pt'
all_keys = list(hists[var].keys())
ttbar_keys = [k for k in all_keys if 'TTToSemiLeptonic' in k[0]]

print(f'Variations available for TTToSemiLeptonic ({var}):')
for k in sorted(ttbar_keys):
    print(' ', k[2])   # variation name

In [ ]:
# Plot nominal vs JES up/down
nom  = hists[var][('TTToSemiLeptonic', 'SR', 'nominal')]

fig, (ax, ax_r) = plt.subplots(2, 1, figsize=(7, 6),
                                gridspec_kw={'height_ratios': [3,1]},
                                sharex=True)

edges   = nom.axes[0].edges
centers = 0.5 * (edges[:-1] + edges[1:])

ax.stairs(nom.values(), edges, label='Nominal', color='black')

for var_name, color, label in [
    ('JES_up',   'red',  'JES up'),
    ('JES_down', 'blue', 'JES down'),
]:
    key = ('TTToSemiLeptonic', 'SR', var_name)
    if key in hists[var]:
        h_var = hists[var][key]
        ax.stairs(h_var.values(), edges, label=label, color=color, linestyle='--')
        ratio = np.where(nom.values() > 0, h_var.values() / nom.values(), 1.0)
        ax_r.stairs(ratio, edges, color=color, linestyle='--')

ax_r.axhline(1, color='black', linewidth=0.8)
ax_r.set_ylim(0.8, 1.2)
ax_r.set_ylabel('Var / Nom')
ax.set_ylabel('Events / bin')
ax.legend()
hep.cms.label('Simulation Preliminary', ax=ax)
plt.tight_layout()

## Dataset summary from CSV

In [ ]:
df = pd.read_csv('../datasets/dataset_summary.csv')
display(df[~df.is_data][['sample', 'process_group', 'n_events',
                          'cross_section_pb', 'expected_yield']]
          .set_index('sample')
          .style.format({'n_events': '{:,.0f}',
                         'cross_section_pb': '{:.3g}',
                         'expected_yield': '{:,.0f}'}))